# Example 1: PDF 테이블 정보에 대한 Recursive Retrieval 전략
- 다수의 CSV 테이블 대상으로 검색 chunk와 답변 생성 chunk 분리해보기

In [1]:
# pip install llama-index-embeddings-openai llama-index-llms-openai camelot-py llama-index

In [2]:
# !apt-get install ghostscript
# !pip install ghostscript

In [3]:
import camelot

# https://en.wikipedia.org/wiki/The_World%27s_Billionaires
from llama_index.core import VectorStoreIndex
# from llama_index.core.query_engine import PandasQueryEngine
from llama_index.core.schema import IndexNode
from llama_index.llms.openai import OpenAI

from llama_index.readers.file import PyMuPDFReader
from typing import List

/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [4]:
from dotenv import load_dotenv
import os
import openai
load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]

In [5]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

# 추후 사용할 llm, 임베딩 모델 클래스 정의
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small", dimensions=512)


/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# 파싱할 파일 경로 설정
file_path = "dataset/Billionaires.pdf"

In [7]:
# PDF파서 정의
reader = PyMuPDFReader()

In [8]:
# 업로드된 경로에서 로딩스테이지 진행한 후 다큐먼트 단위로 저장
docs = reader.load(file_path)

In [9]:
# 도큐먼트 정보 확인
docs

[Document(id_='28a24f0d-d7eb-45a7-b5ab-15d631cc3f97', embedding=None, metadata={'total_pages': 35, 'file_path': 'dataset/Billionaires.pdf', 'source': '1'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text="The World's Billionaires\nList of the world's billionaires, ranked in order of net worth\nThe net worth of the world's billionaires increased from\nless than $1 trillion in 2000 to over $16 trillion in 2025.\nPublication details\nPublisher\nWhale Media Investments\nForbes family\nPublication\nForbes\nFirst published\nMarch 1987[1]\nLatest publication\nApril 1, 2025\n2025 list details[2]\nWealthiest\nElon Musk\nNet worth (1st)\n\xa0US$342\xa0billion\nNumber of\nbillionaires\n\xa03,028 (from 2,781)\nTotal list net worth\nvalue\n\xa0US$16.1\xa0trillion (from US$14.2\ntrillion)\nNumber of women\n\xa0406\nNumber of men\n\xa02,622\nNew m

In [10]:
from llama_index.core import Settings
#노드변환 및 파싱
doc_nodes = Settings.node_parser.get_nodes_from_documents(docs)

In [11]:
doc_nodes

[TextNode(id_='31d8f268-e35f-42f8-b676-bae5aa29ff6d', embedding=None, metadata={'total_pages': 35, 'file_path': 'dataset/Billionaires.pdf', 'source': '1'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='28a24f0d-d7eb-45a7-b5ab-15d631cc3f97', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'total_pages': 35, 'file_path': 'dataset/Billionaires.pdf', 'source': '1'}, hash='290cadf340323aac5ec3699cb86bcaa607560cedb39511ae0774637bb885f91e')}, metadata_template='{key}: {value}', metadata_separator='\n', text="The World's Billionaires\nList of the world's billionaires, ranked in order of net worth\nThe net worth of the world's billionaires increased from\nless than $1 trillion in 2000 to over $16 trillion in 2025.\nPublication details\nPublisher\nWhale Media Investments\nForbes family\nPublication\nForbes\nFirst published\nMarch 1987[1]\nLatest publication\nApril 1, 2025\n2025 list details[2]\nWealthiest\n

In [12]:
vector_index0 = VectorStoreIndex(doc_nodes)
vector_query_engine0 = vector_index0.as_query_engine()

2026-02-28 08:45:03,088 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [13]:
vector_index0

In [14]:
response = vector_query_engine0.query(
    "How many billionaires were there in 2009?"
)

2026-02-28 08:45:05,394 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-28 08:45:06,776 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [15]:
print(response.source_nodes[0].node.get_content())

No.
Name
Net worth
(USD)
Age
Nationality
Source(s) of wealth
1 
Carlos Slim
$74.0 billion 
71
 Mexico
América Móvil, Grupo Carso
2 
Bill Gates
$56.0 billion 
55
 United
States
Microsoft
3 
Warren Buffett
$50.0 billion 
80
 United
States
Berkshire Hathaway
4 
Bernard Arnault
$41.0 billion 
62
 France
LVMH Moët Hennessy • Louis
Vuitton
5 
Larry Ellison
$39.5 billion 
66
 United
States
Oracle Corporation
6 
Lakshmi Mittal
$31.1 billion 
60
 India
Arcelor Mittal
7 
Amancio Ortega
$31.0 billion 
74
 Spain
Inditex Group
8 
Eike Batista
$30.0 billion 
53
 Brazil
EBX Group
9 
Mukesh Ambani
$27.0 billion 
54
 India
Reliance Industries
10
Christy Walton &
family
$26.5 billion 
62
 United
States
Walmart
Slim narrowly eclipsed Gates to top the billionaire list for the first time. Slim saw his estimated worth
surge $18.5 billion to $53.5 billion as shares of America Movil rose 35 percent. Gates' estimated wealth
rose $13 billion to $53 billion, placing him second. Warren Buffett was third with $47 

In [16]:
print(str(response))

The information provided does not specify the number of billionaires in 2009.


In [17]:
response = vector_query_engine0.query(
    "What's the net worth of the second richest billionaire in 2023?"
)
print(str(response))

2026-02-28 08:45:06,911 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-28 08:45:08,915 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The net worth of the second richest billionaire in 2023 is $216 billion.


In [18]:
response

Response(response='The net worth of the second richest billionaire in 2023 is $216 billion.', source_nodes=[NodeWithScore(node=TextNode(id_='6b377d92-c09a-41a3-96fe-8dc5b928b9ce', embedding=None, metadata={'total_pages': 35, 'file_path': 'dataset/Billionaires.pdf', 'source': '4'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='62f2b8c8-0893-4082-b48a-6fdc7c5b7f1c', node_type='4', metadata={'total_pages': 35, 'file_path': 'dataset/Billionaires.pdf', 'source': '4'}, hash='d44df096d348872a3f1cea025a1e524c576f4fc93ba0cf998ed0050c0cf8bf37')}, metadata_template='{key}: {value}', metadata_separator='\n', text="No.\nName\nNet worth\n(USD)\nAge\nNationality\nPrimary source(s) of\nwealth\n1 \nBernard Arnault &\nfamily\n$233\xa0billion\xa0\n75\n\xa0France\nLVMH\n2 \nElon Musk\n$195\xa0billion\xa0\n52\n\xa0South Africa\n\xa0Canada\n\xa0United\nStates\nTesla, SpaceX\n3 \nJeff Bezos\n$194\xa0billion\xa0\n60\n\xa

In [19]:
print(response.source_nodes[0].node.get_content())

No.
Name
Net worth
(USD)
Age
Nationality
Primary source(s) of
wealth
1 
Bernard Arnault &
family
$233 billion 
75
 France
LVMH
2 
Elon Musk
$195 billion 
52
 South Africa
 Canada
 United
States
Tesla, SpaceX
3 
Jeff Bezos
$194 billion 
60
 United
States
Amazon
4 
Mark Zuckerberg
$177 billion 
39
 United
States
Meta Platforms
5 
Larry Ellison
$141 billion 
79
 United
States
Oracle Corporation
6 
Warren Buffett
$133 billion 
93
 United
States
Berkshire Hathaway
7 
Bill Gates
$128 billion 
68
 United
States
Microsoft
8 
Steve Ballmer
$121 billion 
68
 United
States
Microsoft
9 
Mukesh Ambani
$116 billion 
66
 India
Reliance Industries
10
Larry Page
$114 billion 
51
 United
States
Google
In the 37th annual Forbes list of the world's billionaires, the list included 2,640 billionaires with a total
net wealth of $12.2 trillion, down 28 members and $500 billion from 2022. Over half of the list is less
wealthy compared to the previous year, including Elon Musk, who fell from No. 1 to No. 2.[8] 

- 기본적인 PDF파싱모듈로는 테이블 등 Text-Only 가 아닌 문서에 대한 정보 해석력이 떨어지는 것을 확인
- Table정보를 따로 추출하여 답하는 방식은 어떨지?

In [54]:
def normalize_columns(df):
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace("\n", " ")
        .str.replace(r"\s+", " ", regex=True)
        .str.lower()
    )
    return df

In [55]:
def get_tables(path: str, pages: List[int]):
    table_dfs = []

    for page in pages:
        tables = camelot.read_pdf(path, pages=str(page))

        for table in tables:
            df = table.df.copy()

            # 첫 행을 header로 쓰되, 비정상일 수 있음을 전제로
            header = df.iloc[0]
            df = df.iloc[1:].reset_index(drop=True)
            df.columns = header

            # 🔴 핵심: 컬럼 정규화
            df = normalize_columns(df)

            table_dfs.append(df)

    return table_dfs

In [56]:
table_dfs = get_tables(file_path, pages=[3,4,24])

KeyError: 'year'

In [58]:
#파싱된 테이블 개수확인
len(table_dfs)

4

In [59]:
#파싱 결과 확인
table_dfs[0]

,no.,name,net worth (usd),age,nationality,primary source(s) of wealth
0,1,Elon Musk,$342 billion,53,South Africa\n Canada\n United\nStates,Tesla and SpaceX
1,2,Mark Zuckerberg[nb 1],$216 billion,40,United\nStates,Meta Platforms
...,...,...,...,...,...,...
8,9,Amancio Ortega,$124 billion,89,Spain,"Inditex, Zara"
9,10,Steve Ballmer,$118 billion,69,United\nStates,Microsoft


In [24]:
#파싱 결과 확인
table_dfs[1]

,No.,Name,Net worth\n(USD),Age,Nationality,Primary source(s) of\nwealth
0,1,Bernard Arnault &\nfamily,$233 billion,75,France,LVMH
1,2,Elon Musk,$195 billion,52,South Africa\n Canada\n United\nStates,"Tesla, SpaceX"
2,3,Jeff Bezos,$194 billion,60,United\nStates,Amazon
3,4,Mark Zuckerberg,$177 billion,39,United\nStates,Meta Platforms
4,5,Larry Ellison,$141 billion,79,United\nStates,Oracle Corporation
5,6,Warren Buffett,$133 billion,93,United\nStates,Berkshire Hathaway
6,7,Bill Gates,$128 billion,68,United\nStates,Microsoft
7,8,Steve Ballmer,$121 billion,68,United\nStates,Microsoft
8,9,Mukesh Ambani,$116 billion,66,India,Reliance Industries
9,10,Larry Page,$114 billion,51,United\nStates,Google


In [25]:
#파싱 결과 확인
table_dfs[-1]

,No.[50],Name,Net worth (USD),Nationality,Source(s) of wealth
0,1,Walton family,$18.5 billion,United States,Wal-Mart
1,2,Taikichiro Mori,$15.0 billion,Japan,Mori Building Company
2,3,Yoshiaki Tsutsumi,$14.0 billion,Japan,Seibu Corporation
3,4,du Pont family,$10.0 billion,United States,DuPont
4,5,Hans and Gad Rausing,$9.0 billion,Sweden,Tetra Pak
5,6,Kitaro Watanabe,$7.7 billion,Japan,Azabu Building
6,7,Paul Reichmann & brothers,$7.1 billion,Canada,Olympia & York
7,8,Kenneth Thomson,$6.8 billion,Canada,Thomson Corporation
8,9,Kenkichi Nakajima,$6.1 billion,Japan,Heiwa Corporation
9,10,Shin Kyuk-ho,$6.0 billion,South Korea,Lotte Corporation


이제 테이블을 다 파싱해왔는데,
이것들을 기반으로 질문에 바로 답할수 있도록 만들려면 만들수도 있겠지만,
테이블이 지금과 다르게 수천 수만개일때, 모든 유저 쿼리에 대해 수만개의 테이블을 매번 조회하는 것은 실용성 없는 Naive한 접근방식(자원은 무한하지 않다).

그렇기 때문에,
1. 사용자의 질문과 관련된 테이블을 먼저 찾고
2. 찾은 테이블을 기준으로 사용자의 질문에 답할 수 있는 정보를 발췌하여 답해보자.

일단은 각 테이블별로 답해주는 담당 라마인덱스 쿼리엔진을 만들어주자.

In [26]:
from llama_index.experimental.query_engine import PandasQueryEngine

llm = OpenAI(model="gpt-4o-mini", temperature=0)

df_query_engines = [
    PandasQueryEngine(table_df, llm=llm) for table_df in table_dfs
]


In [65]:
# 상응하는 테이블 지정해서 답변 요구
response = df_query_engines[0].query(
    "What's the net worth of the second richest billionaire in 2023?"
)
print(response)

2026-02-28 10:23:20,206 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
There was an error running the output as Python code. Error message: could not convert string to float: '342 billion'


Traceback (most recent call last):
  File "/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/llama_index/experimental/query_engine/pandas/output_parser.py", line 62, in default_output_processor
    output_str = str(safe_eval(module_end_str, global_vars, local_vars))
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/llama_index/experimental/exec_utils.py", line 259, in safe_eval
    return eval(__source, _get_restricted_globals(__globals), __locals)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/pandas/core/generic.py", line 6643, in astype
    new_data = self._mgr.astype(dtype=dtype, copy=copy, errors=errors)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "

In [28]:
# 상응하는 테이블 지정해서 답변 요구
response = df_query_engines[1].query(
    "What's the net worth of the second richest billionaire in 2022?"
)
print(str(response))

2026-02-28 08:45:17,428 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
There was an error running the output as Python code. Error message: index 0 is out of bounds for axis 0 with size 0


Traceback (most recent call last):
  File "/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/llama_index/experimental/query_engine/pandas/output_parser.py", line 62, in default_output_processor
    output_str = str(safe_eval(module_end_str, global_vars, local_vars))
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/llama_index/experimental/exec_utils.py", line 259, in safe_eval
    return eval(__source, _get_restricted_globals(__globals), __locals)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
IndexError: index 0 is out of bounds for axis 0 with size 0


In [29]:
len(df_query_engines)

4

In [30]:
response = df_query_engines[3].query(
    "How many billionaires were there in 2009?"
)
print(str(response))

2026-02-28 08:45:18,931 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
10


질문별로 담당하는 쿼리엔진을 부여하는 것으로 heuristic하게 서칭 스페이스를 줄이고 시작할 수 있는 것 확인

In [ ]:
# 쿼리엔진 요약문 생성 (엔진 개수와 항상 일치)
summaries = [
    f"This node provides information for extracted billionaire table #{idx}."
    for idx in range(len(df_query_engines))
]

#생성된 요약문 별 노드단위 생성
df_nodes = [
    IndexNode(text=summary, index_id=f"pandas{idx}") for idx, summary in enumerate(summaries)
]

#요약노드 <-> 쿼리엔진 매핑
df_id_query_engine_mapping = {
    f"pandas{idx}": df_query_engine
    for idx, df_query_engine in enumerate(df_query_engines)
}

assert len(df_nodes) == len(df_id_query_engine_mapping), (
    len(df_nodes), len(df_id_query_engine_mapping)
)

In [32]:
#생성된 노드 확인
df_nodes[0]

IndexNode(id_='675c1d2a-9293-495d-b48c-0754016d9930', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='This node provides information for extracted billionaire table #0.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', index_id='pandas0', obj=None)

In [33]:
#상위레벨 벡터스토어인덱스 정의
vector_index = VectorStoreIndex(df_nodes)
vector_retriever = vector_index.as_retriever(similarity_top_k=1)

2026-02-28 08:45:19,740 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [34]:
from llama_index.core.retrievers import RecursiveRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core import get_response_synthesizer

recursive_retriever = RecursiveRetriever(
    "vector",
    retriever_dict={"vector": vector_retriever},
    query_engine_dict=df_id_query_engine_mapping,
    verbose=True,
)

response_synthesizer = get_response_synthesizer(response_mode="compact")

query_engine = RetrieverQueryEngine.from_args(
    recursive_retriever, response_synthesizer=response_synthesizer
)

In [35]:
response = query_engine.query(
    "What's the net worth of the second richest billionaire in 2023?"
)

2026-02-28 08:45:19,897 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Retrieving with query id None: What's the net worth of the second richest billionaire in 2023?
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Retrieved node with id, entering: pandas2
Retrieving with query id pandas2: What's the net worth of the second richest billionaire in 2023?


2026-02-28 08:45:21,417 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Traceback (most recent call last):
  File "/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/llama_index/experimental/query_engine/pandas/output_parser.py", line 62, in default_output_processor
    output_str = str(safe_eval(module_end_str, global_vars, local_vars))
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/llama_index/experimental/exec_utils.py", line 259, in safe_eval
    return eval(__source, _get_restricted_globals(__globals), __locals)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/pandas/core/generic.py", line 6643, in astype
    new_data = self._mgr.astype(dtype=dtype, copy=copy, errors=errors)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "

Got response: There was an error running the output as Python code. Error message: could not convert string to float: '$23.8'


2026-02-28 08:45:22,961 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [36]:
response.source_nodes[0].node.get_content()

"Query: What's the net worth of the second richest billionaire in 2023?\nResponse: There was an error running the output as Python code. Error message: could not convert string to float: '$23.8'"

In [37]:
str(response)

'The net worth of the second richest billionaire in 2023 is $23.8 billion.'

In [38]:
response = query_engine.query("How many billionaires were there in 2009?")

2026-02-28 08:45:23,123 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Retrieving with query id None: How many billionaires were there in 2009?
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Retrieved node with id, entering: pandas1
Retrieving with query id pandas1: How many billionaires were there in 2009?


2026-02-28 08:45:24,387 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Traceback (most recent call last):
  File "/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/pandas/core/indexes/base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'Year'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/llama_index/experimental/query_engine/pandas/output_parser.py", line 62, in default_output_processor
    output_str = str(safe_eval(module_end_str

Got response: There was an error running the output as Python code. Error message: 'Year'


2026-02-28 08:45:25,140 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [39]:
str(response)

'There was an error in retrieving the information regarding the number of billionaires in 2009.'

- 서머리 텍스트로 recursive retriever 모듈로 하여금 우리가 찾고자 하는 문서를 자동으로 판별해서 해당 쿼리엔진을 기반으로만 답하게 하는 것이 가능한 것 확인

- 파인콘 DB에 있는 데이터를 연계시켜서도 가능할까?

In [40]:
!pip install -U llama-index-vector-stores-pinecone pinecone datasets

  Using cached pinecone-8.1.0-py3-none-any.whl.metadata (14 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.6.0
    Uninstalling datasets-4.6.0:
      Successfully uninstalled datasets-4.6.0


In [41]:
import os
import re
from time import sleep

from pinecone import Pinecone, ServerlessSpec

from llama_index.core import (
    Settings,
    Document,
    VectorStoreIndex,
    StorageContext,
    get_response_synthesizer,
)
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.schema import TextNode, IndexNode, MetadataMode
from llama_index.core.retrievers import RecursiveRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.agent import AgentWorkflow
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.pinecone import PineconeVectorStore


In [42]:
from dotenv import load_dotenv

load_dotenv()
pinecone_api_key = os.environ["PINECONE_API_KEY"]
# openai_api_key = os.environ["OPENAI_API_KEY"]

embed_model = OpenAIEmbedding(model="text-embedding-3-small", dimensions=512)
llm = OpenAI(model="gpt-4o-mini", temperature=0)

Settings.embed_model = embed_model
Settings.llm = llm


In [43]:
from datasets import load_dataset

FAST_DEV_MODE = True
DEV_SAMPLE_SIZE = 300
FULL_SAMPLE_EXPR = "train[:1000]"

split_expr = f"train[:{DEV_SAMPLE_SIZE}]" if FAST_DEV_MODE else FULL_SAMPLE_EXPR
dataset = load_dataset("lcw99/wikipedia-korean-20221001", split=split_expr)
data = dataset.to_pandas()[["id", "text", "title"]].drop_duplicates(subset="text", keep="first")
print(f"Loaded rows: {len(data)} (split={split_expr})")


2026-02-28 08:45:37,516 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/lcw99/wikipedia-korean-20221001/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"


HTTP Request: HEAD https://huggingface.co/datasets/lcw99/wikipedia-korean-20221001/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"


2026-02-28 08:45:37,527 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/lcw99/wikipedia-korean-20221001/49af0d4be8266217cb60bbc2bdb8e0ca0929b250/README.md "HTTP/1.1 200 OK"


HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/lcw99/wikipedia-korean-20221001/49af0d4be8266217cb60bbc2bdb8e0ca0929b250/README.md "HTTP/1.1 200 OK"


2026-02-28 08:45:37,728 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/lcw99/wikipedia-korean-20221001/resolve/49af0d4be8266217cb60bbc2bdb8e0ca0929b250/wikipedia-korean-20221001.py "HTTP/1.1 404 Not Found"


...

HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=lcw99/wikipedia-korean-20221001 "HTTP/1.1 200 OK"


2026-02-28 08:45:39,405 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/lcw99/wikipedia-korean-20221001/tree/49af0d4be8266217cb60bbc2bdb8e0ca0929b250/data?recursive=true&expand=false "HTTP/1.1 200 OK"


HTTP Request: GET https://huggingface.co/api/datasets/lcw99/wikipedia-korean-20221001/tree/49af0d4be8266217cb60bbc2bdb8e0ca0929b250/data?recursive=true&expand=false "HTTP/1.1 200 OK"


2026-02-28 08:45:39,601 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/lcw99/wikipedia-korean-20221001/tree/49af0d4be8266217cb60bbc2bdb8e0ca0929b250?recursive=false&expand=false "HTTP/1.1 200 OK"


HTTP Request: GET https://huggingface.co/api/datasets/lcw99/wikipedia-korean-20221001/tree/49af0d4be8266217cb60bbc2bdb8e0ca0929b250?recursive=false&expand=false "HTTP/1.1 200 OK"
Loaded rows: 300 (split=train[:300])


In [44]:
def clean_up_text(content: str) -> str:
    content = re.sub(r"(\w+)-\n(\w+)", r"\1\2", content)
    content = re.sub(r"\\n|  —|——————————|—————————|—————|\\u[\dA-Fa-f]{4}|\uf075|\uf0b7", "", content)
    content = re.sub(r"(\w)\s*-\s*(\w)", r"\1-\2", content)
    content = re.sub(r"\s+", " ", content)
    return content

documents = [
    Document(
        text=clean_up_text(row["text"]),
        doc_id=str(row["id"]),
        metadata={"title": str(row["title"])}
    )
    for _, row in data.iterrows()
]


In [45]:
data.head()

id  \
0   5   
1   9   
2  10   
3  19   
4  20   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [46]:
pc = Pinecone(api_key=pinecone_api_key)

index_name = "vector"
target_dimension = 512
metric = "dotproduct"
spec = ServerlessSpec(cloud="aws", region="us-east-1")

existing = pc.list_indexes().names()

if index_name in existing:
    desc = pc.describe_index(index_name)
    current_dimension = desc["dimension"] if isinstance(desc, dict) else desc.dimension

    if current_dimension != target_dimension:
        raise ValueError(
            f"Existing index '{index_name}' dimension is {current_dimension}, "
            f"but target is {target_dimension}. "
            "Delete/recreate index or use a different index name."
        )
else:
    pc.create_index(
        name=index_name,
        dimension=target_dimension,
        metric=metric,
        spec=spec,
    )

while not pc.describe_index(index_name).status["ready"]:
    sleep(1)

pinecone_index = pc.Index(index_name)
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print(pinecone_index.describe_index_stats())


{'dimension': 512,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 7140}},
 'total_vector_count': 7140,
 'vector_type': 'dense'}


In [47]:
# from llama_index.core import VectorStoreIndex
# from llama_index.core.node_parser import TokenTextSplitter
# from llama_index.core.schema import TextNode, MetadataMode

# # 병목 완화: 청크 수 감소 + 배치 확대 + 비동기
# CHUNK_SIZE = 1500
# CHUNK_OVERLAP = 150
# INSERT_BATCH_SIZE = 300
# RETRIEVAL_PROXY_CHARS = 2200

# splitter = TokenTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
# nodes = splitter.get_nodes_from_documents(documents)

# safe_nodes = []
# for i, n in enumerate(nodes):
#     full_text = n.get_content(metadata_mode=MetadataMode.NONE)
#     proxy_text = full_text[:RETRIEVAL_PROXY_CHARS]

#     safe_nodes.append(
#         TextNode(
#             text=proxy_text,
#             id_=f"node-{i}",
#             metadata={"title": n.metadata.get("title", "")},
#         )
#     )

# print(f"chunks: {len(nodes)} -> indexed proxy chunks: {len(safe_nodes)}")

# vs_index = VectorStoreIndex(
#     nodes=safe_nodes,
#     storage_context=storage_context,
#     embed_model=embed_model,
#     insert_batch_size=INSERT_BATCH_SIZE,
#     use_async=True,
#     show_progress=True,
# )


In [48]:
stats = pinecone_index.describe_index_stats()
print(stats)


{'dimension': 512,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 7140}},
 'total_vector_count': 7140,
 'vector_type': 'dense'}


In [49]:
sample_ids = []
for ids in pinecone_index.list(namespace=""):
    sample_ids.extend(ids)
    if len(sample_ids) >= 5:
        break

sample_ids = sample_ids[:5]
print("sample_ids:", sample_ids)

fetched = pinecone_index.fetch(ids=sample_ids, namespace="")
print(fetched)


sample_ids: ['node-0', 'node-1', 'node-10', 'node-100', 'node-1000']
FetchResponse(namespace='', vectors={'node-100': Vector(id='node-100', values=[-0.00739439297, -0.0128739635, -0.0178615842, -0.0341186523, 0.0561277568, -0.0404761657, -0.0682978556, 0.0133205028, 0.040990822, -0.0616073273, -0.0462887473, 0.0406275354, -0.0527673587, -0.0121095488, -0.0371157676, -0.0371460393, -0.0172258317, -0.0134718725, 0.00815124, -0.014985566, -0.0175891183, -0.047984086, -0.0356323458, 0.00724680768, 0.0548259802, 0.0242342334, -0.00316172745, 0.0641200617, 0.00108796719, 0.00650888216, -0.0270042922, -0.0512536652, -0.00949085876, -0.0450475216, -0.00346257398, 0.0107623609, 0.0525857136, -0.0410816446, 0.0121322535, -0.0225237608, -0.0369946696, 0.036903847, 0.0408394523, -0.0274432637, 0.00890808646, 0.0241131373, 0.0345122144, 0.0350268669, -0.00764036831, -0.0159846041, -0.032120578, 0.0261566248, -0.0870676562, -0.0039583086, -0.020162398, 0.00669052545, -0.0699931905, 0.0239769053, 0.0

In [50]:
qvec = embed_model.get_text_embedding("테스트 질의")
res = pinecone_index.query(
    vector=qvec,
    top_k=5,
    include_values=False,
    include_metadata=True,
    namespace="",
)
print(res)


2026-02-28 08:45:45,739 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
{'matches': [{'id': 'node-1797',
              'metadata': {'_node_content': '{"id_": "node-1797", "embedding": '
                                            'null, "metadata": {}, '
                                            '"excluded_embed_metadata_keys": '
                                            '[], "excluded_llm_metadata_keys": '
                                            '[], "relationships": {}, '
                                            '"metadata_template": "{key}: '
                                            '{value}", "metadata_separator": '
                                            '"\\n", "text": "랩톤 그리고 끈이론. 2015년 '
                                            '6월 10일. 기본 입자 페르미 입자 양자색역학 피네간의 '
                                            '경야", "mimetype": "text/plain", '
                                            '"start_char_idx": null, '
                                            '"e

In [51]:
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model,
)
index_stats = pinecone_index.describe_index_stats()
index_stats


{'dimension': 512,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 7140}},
 'total_vector_count': 7140,
 'vector_type': 'dense'}

In [52]:
from llama_index.core import VectorStoreIndex
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.agent import AgentWorkflow
from llama_index.llms.openai import OpenAI

# pc = Pinecone(api_key=...)
# # pc_index = pc.Index(index_name)  # 기존 인덱스
# vector_store = PineconeVectorStore(pinecone_index=pc_index)

index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

OPENAI_MODEL = "gpt-4o-mini"

agents = {}
for title in data["title"].dropna().unique():
    vector_query_engine = index.as_query_engine(
        vector_store_kwargs={"filter": {"title": title}}
    )

    query_engine_tools = [
        QueryEngineTool(
            query_engine=vector_query_engine,
            metadata=ToolMetadata(
                name="vector_tool",
                description=f"{title}에 대해서 물어볼 때 사용"
            ),
        ),
    ]

    function_llm = OpenAI(model=OPENAI_MODEL, temperature=0)
    agent = AgentWorkflow.from_tools_or_functions(
        tools_or_functions=query_engine_tools,
        llm=function_llm,
        verbose=True,
    )

    agents[title] = agent

In [53]:
handler = agents["지미 카터"].run(user_msg="이 인물 핵심 요약")
result = await handler
print(result.response.content)

2026-02-28 08:45:47,651 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-02-28 08:45:48,041 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-02-28 08:45:53,985 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-02-28 08:45:54,466 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


CancelledError: 

In [ ]:
#생성된 에이전트 확인
agents

{'지미 카터': <llama_index.core.agent.workflow.multi_agent_workflow.AgentWorkflow at 0x13920e870>,
 '수학': <llama_index.core.agent.workflow.multi_agent_workflow.AgentWorkflow at 0x1398d95e0>,
 '수학 상수': <llama_index.core.agent.workflow.multi_agent_workflow.AgentWorkflow at 0x13a3e7440>,
 '문학': <llama_index.core.agent.workflow.multi_agent_workflow.AgentWorkflow at 0x13a3e6f90>,
 '나라 목록': <llama_index.core.agent.workflow.multi_agent_workflow.AgentWorkflow at 0x13a3e52e0>,
 '화학': <llama_index.core.agent.workflow.multi_agent_workflow.AgentWorkflow at 0x13a3e7c20>,
 '체첸 공화국': <llama_index.core.agent.workflow.multi_agent_workflow.AgentWorkflow at 0x13a3e6fc0>,
 '맥스웰 방정식': <llama_index.core.agent.workflow.multi_agent_workflow.AgentWorkflow at 0x13a3e6510>,
 '초월수': <llama_index.core.agent.workflow.multi_agent_workflow.AgentWorkflow at 0x13a3e5b80>,
 '음계': <llama_index.core.agent.workflow.multi_agent_workflow.AgentWorkflow at 0x139e587d0>,
 '대한민국 제16대 대통령 선거': <llama_index.core.agent.workflow.multi_a

In [ ]:
nodes = []
for title in data["title"].dropna().unique():
    doc_summary = (
        f"이것은 {title}과 관련된 내용이 있습니다. "
        f"{title}과 관련된 내용을 확인하는 용도로 이 인덱스를 사용하세요."
    )
    node = IndexNode(text=doc_summary, index_id=title)
    nodes.append(node)


In [ ]:
title_query_engines = {}
for title in data["title"].dropna().unique():
    title_query_engines[title] = index.as_query_engine(
        vector_store_kwargs={"filter": {"title": title}}
    )

vector_index = VectorStoreIndex(nodes, embed_model=embed_model)
vector_retriever = vector_index.as_retriever(similarity_top_k=1)

recursive_retriever = RecursiveRetriever(
    "vector",
    retriever_dict={"vector": vector_retriever},
    query_engine_dict=title_query_engines,
    verbose=True,
)

response_synthesizer = get_response_synthesizer(response_mode="compact")

query_engine = RetrieverQueryEngine.from_args(
    recursive_retriever,
    response_synthesizer=response_synthesizer,
)


2026-02-26 17:25:45,970 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-02-26 17:25:46,188 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-02-26 17:25:46,462 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [ ]:
# 해당 문서에서만 답변 가능한 굉장히 구체적인 질문 테스트
response = query_engine.query("셀빅에 대해 알려줘")

2026-02-26 17:25:46,648 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Retrieving with query id None: 셀빅에 대해 알려줘
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Retrieved node with id, entering: 셀빅
Retrieving with query id 셀빅: 셀빅에 대해 알려줘


2026-02-26 17:25:49,402 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Got response: 셀빅은 대한민국에서 생산된 PDA의 일종으로, jTel에서 개발하였으며 후에 코오롱에 인수되어 Cellvic으로 이름이 변경되었습니다. 이 제품은 팜 파일럿을 벤치마크하여 한국 환경에 맞춘 자체 운영 체제를 갖추고 있으며, 한국 최초의 PDA OS로 중요한 의미를 지닙니다. jTel은 애플리케이션 공모전을 통해 다양한 애플리케이션을 확보하고, 개인 개발자들을 지원하여 많은 애플리케이션이 개발되었습니다. 셀빅은 한글 지원이 원활하며, 다양한 모델이 출시되었으나, 2004년부터 후속 지원이 중단되었습니다.


2026-02-26 17:25:51,910 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [ ]:
response

Response(response='셀빅은 대한민국에서 개발된 PDA로, jTel에서 시작하여 코오롱에 인수된 후 Cellvic으로 이름이 변경되었습니다. 이 제품은 한국 환경에 맞춘 자체 운영 체제를 갖추고 있으며, 한국 최초의 PDA OS로 중요한 의미를 지닙니다. jTel은 애플리케이션 공모전을 통해 다양한 애플리케이션을 확보하고 개인 개발자들을 지원하여 많은 애플리케이션이 개발되었습니다. 셀빅은 한글 지원이 원활하며, 여러 모델이 출시되었으나 2004년부터 후속 지원이 중단되었습니다.', source_nodes=[NodeWithScore(node=TextNode(id_='358d51f6-443e-411b-91f1-d2700e08389c', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Query: 셀빅에 대해 알려줘\nResponse: 셀빅은 대한민국에서 생산된 PDA의 일종으로, jTel에서 개발하였으며 후에 코오롱에 인수되어 Cellvic으로 이름이 변경되었습니다. 이 제품은 팜 파일럿을 벤치마크하여 한국 환경에 맞춘 자체 운영 체제를 갖추고 있으며, 한국 최초의 PDA OS로 중요한 의미를 지닙니다. jTel은 애플리케이션 공모전을 통해 다양한 애플리케이션을 확보하고, 개인 개발자들을 지원하여 많은 애플리케이션이 개발되었습니다. 셀빅은 한글 지원이 원활하며, 다양한 모델이 출시되었으나, 2004년부터 후속 지원이 중단되었습니다.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}'), score=0.5142472164885874)], metad

In [ ]:
response.response

'셀빅은 대한민국에서 개발된 PDA로, jTel에서 시작하여 코오롱에 인수된 후 Cellvic으로 이름이 변경되었습니다. 이 제품은 한국 환경에 맞춘 자체 운영 체제를 갖추고 있으며, 한국 최초의 PDA OS로 중요한 의미를 지닙니다. jTel은 애플리케이션 공모전을 통해 다양한 애플리케이션을 확보하고 개인 개발자들을 지원하여 많은 애플리케이션이 개발되었습니다. 셀빅은 한글 지원이 원활하며, 여러 모델이 출시되었으나 2004년부터 후속 지원이 중단되었습니다.'

In [ ]:
# Response 객체 내부 key
print(response.__dict__.keys())

# metadata 딕셔너리 key
print(response.metadata.keys())

# source_nodes 각 노드 metadata key
for i, n in enumerate(response.source_nodes):
    print(i, n.node.metadata.keys())

dict_keys(['response', 'source_nodes', 'metadata'])
dict_keys(['358d51f6-443e-411b-91f1-d2700e08389c'])
0 dict_keys([])


In [ ]:
response = query_engine.query("한국 강의 종류에 대해서 알려줘")

2026-02-26 17:25:52,093 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Retrieving with query id None: 한국 강의 종류에 대해서 알려줘
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Retrieved node with id, entering: 한국
Retrieving with query id 한국: 한국 강의 종류에 대해서 알려줘


2026-02-26 17:25:54,173 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Got response: 한국의 강은 다양한 종류가 있으며, 주요 강으로는 한강, 낙동강, 금강, 영산강 등이 있습니다. 이들 강은 각각의 지역에서 중요한 역할을 하며, 수자원 공급, 농업, 교통 등 다양한 용도로 사용됩니다. 한강은 서울을 가로지르며, 낙동강은 경상도를 흐르고, 금강은 충청도를 지나며, 영산강은 전라도를 흐릅니다. 각 강은 그 지역의 생태계와 문화에도 큰 영향을 미치고 있습니다.


2026-02-26 17:25:56,857 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [ ]:
response.response

'한국의 강은 여러 종류가 있으며, 주요 강으로는 한강, 낙동강, 금강, 영산강 등이 있습니다. 이들 강은 각 지역에서 중요한 역할을 하며, 수자원 공급, 농업, 교통 등 다양한 용도로 사용됩니다. 한강은 서울을 가로지르고, 낙동강은 경상도를 흐르며, 금강은 충청도를 지나고, 영산강은 전라도를 흐릅니다. 각 강은 그 지역의 생태계와 문화에도 큰 영향을 미치고 있습니다.'

## 실전 운영 전략: 테이블별 에이전트 난립을 피하는 리팩터링

현재 실습은 테이블마다 전용 엔진(에이전트)을 만들어 라우팅하는 개념 검증(POC)으로 매우 좋다.
다만 운영 환경에서 테이블 수가 커질수록(수천~수만) `엔진 객체 수`, `초기화 시간`, `라우팅 오탐`이 빠르게 비용으로 전환된다.

아래 리팩터링은 다음 원칙을 따른다.

1. **Lazy Initialization**
   - 모든 테이블 엔진을 미리 만들지 않고, 실제로 선택된 테이블만 생성/캐시한다.
2. **Top-k Routing**
   - `top_k=1` 고정 대신, 상위 후보 테이블 몇 개를 고른 뒤 최종 답변을 합성한다.
3. **계층형 후보 축소 (Domain -> Table)**
   - 제목/메타데이터를 이용해 도메인 후보를 먼저 좁히고, 그 안에서 테이블 라우팅을 수행한다.
4. **운영 가시성**
   - 어떤 도메인/테이블이 선택됐는지 로그를 남겨 실패 원인을 추적할 수 있게 한다.

> 핵심: **"테이블당 에이전트 고정 생성"에서 "필요할 때만 생성 + 후보 축소" 구조로 전환**


In [ ]:
from typing import Dict, List, Optional
from llama_index.core import Settings, VectorStoreIndex
from llama_index.core.schema import IndexNode

# 사전 조건: 기존 셀에서 data(컬럼: title), index(VectorStoreIndex)가 생성되어 있어야 함
required_vars = ["data", "index"]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise ValueError(f"Missing required variables: {missing}. Run previous cells first.")


class ProductionTableRouter:
    """
    운영형 라우터
    - domain_router: 도메인 후보를 먼저 축소
    - table_router: 선택된 도메인 내에서 테이블 top-k 선택
    - lazy cache: 선택된 테이블 query engine만 생성
    """

    def __init__(
        self,
        base_index: VectorStoreIndex,
        table_titles: List[str],
        domain_rules: Optional[Dict[str, List[str]]] = None,
    ):
        self.base_index = base_index
        self.table_titles = table_titles
        self._engine_cache: Dict[str, object] = {}

        # 규칙 기반 도메인 매핑 (프로덕션에서는 별도 카탈로그/메타스토어 권장)
        self.domain_rules = domain_rules or {
            "finance": ["revenue", "profit", "asset", "stock", "market", "billionaire"],
            "people": ["person", "name", "age", "rank", "ceo", "founder"],
            "geography": ["country", "city", "region", "nation"],
            "other": [],
        }

        self.title_to_domain = {title: self._infer_domain(title) for title in self.table_titles}

        # 1) 도메인 라우터 인덱스
        domain_nodes = []
        for domain, keywords in self.domain_rules.items():
            text = f"domain={domain}; keywords={', '.join(keywords)}"
            domain_nodes.append(IndexNode(text=text, index_id=f"domain::{domain}"))
        self.domain_index = VectorStoreIndex(domain_nodes)

        # 2) 테이블 라우터 인덱스 (짧은 proxy 텍스트만 사용)
        table_nodes = []
        for title in self.table_titles:
            domain = self.title_to_domain[title]
            summary = f"table={title}; domain={domain}; use when query is about {title}"
            table_nodes.append(IndexNode(text=summary, index_id=title))
        self.table_index = VectorStoreIndex(table_nodes)

    def _infer_domain(self, title: str) -> str:
        t = str(title).lower()
        for domain, keywords in self.domain_rules.items():
            if any(k in t for k in keywords):
                return domain
        return "other"

    def _get_engine(self, title: str):
        # lazy init + cache
        if title not in self._engine_cache:
            self._engine_cache[title] = self.base_index.as_query_engine(
                vector_store_kwargs={"filter": {"title": title}}
            )
        return self._engine_cache[title]

    @staticmethod
    def _extract_index_id(node_with_score) -> Optional[str]:
        node = getattr(node_with_score, "node", node_with_score)
        return getattr(node, "index_id", None)

    def _route_domains(self, question: str, top_k_domains: int = 2) -> List[str]:
        retriever = self.domain_index.as_retriever(similarity_top_k=top_k_domains)
        hits = retriever.retrieve(question)
        domains = []
        for h in hits:
            idx = self._extract_index_id(h)
            if idx and idx.startswith("domain::"):
                domains.append(idx.split("::", 1)[1])
        return domains or ["other"]

    def _route_tables(self, question: str, allowed_domains: List[str], top_k_tables: int = 4) -> List[str]:
        # 2단계 라우팅: 전체 라우터 결과를 받은 뒤 도메인으로 필터
        retriever = self.table_index.as_retriever(similarity_top_k=max(top_k_tables * 3, top_k_tables))
        hits = retriever.retrieve(question)

        selected = []
        for h in hits:
            title = self._extract_index_id(h)
            if not title:
                continue
            if self.title_to_domain.get(title, "other") in allowed_domains and title not in selected:
                selected.append(title)
            if len(selected) >= top_k_tables:
                break

        # 도메인 필터 후 비어 있으면 fallback
        if not selected:
            for h in hits:
                title = self._extract_index_id(h)
                if title and title not in selected:
                    selected.append(title)
                if len(selected) >= top_k_tables:
                    break

        return selected

    def query(self, question: str, top_k_domains: int = 2, top_k_tables: int = 3, debug: bool = True):
        routed_domains = self._route_domains(question, top_k_domains=top_k_domains)
        routed_tables = self._route_tables(question, routed_domains, top_k_tables=top_k_tables)

        partial_answers = []
        for title in routed_tables:
            qe = self._get_engine(title)
            try:
                ans = qe.query(question)
                partial_answers.append((title, str(ans)))
            except Exception as e:
                partial_answers.append((title, f"[ERROR] {e}"))

        context = "\n\n".join([f"[table={t}]\n{a}" for t, a in partial_answers])
        final_prompt = (
            "You are a synthesis assistant.\n"
            "Combine the partial answers into one final answer.\n"
            "If conflicts exist, mention them explicitly.\n\n"
            f"Question: {question}\n\n"
            f"Partial Answers:\n{context}"
        )

        final = Settings.llm.complete(final_prompt)
        final_text = getattr(final, "text", str(final))

        if debug:
            print("routed_domains:", routed_domains)
            print("routed_tables:", routed_tables)
            print("cached_engines:", len(self._engine_cache))

        return {
            "answer": final_text,
            "routed_domains": routed_domains,
            "routed_tables": routed_tables,
            "partial_answers": partial_answers,
        }


# ===== Usage =====
titles = [t for t in data["title"].dropna().unique().tolist()]
router = ProductionTableRouter(base_index=index, table_titles=titles)

result = router.query(
    "Who is the richest person and what country are they from?",
    top_k_domains=2,
    top_k_tables=3,
    debug=True,
)

print(result["answer"])


## 현재 노트북 전략 평가 (강점/한계/개선점)

### 강점
- **검색 청크와 답변 청크 분리**라는 핵심 아이디어를 정확히 구현함
- PDF 본문 파싱 한계를 테이블 추출(`camelot`) + 전용 질의엔진으로 보완함
- `RecursiveRetriever` 기반 라우팅으로 검색 범위를 줄여 비용 절감 가능성을 보임

### 한계
- 테이블 수가 증가하면 `테이블별 엔진` 객체 관리 비용이 급증
- 라우팅 프록시가 단순 요약문 위주라 오탐/누락 가능
- `top_k=1` 중심 전략은 누락 리스크가 큼
- 운영 관점 메타데이터(품질, 최신성, 소유자, 보안등급, 시간범위) 미반영

### 개선점
1. **Ingestion 표준화**: 원본별 파싱 품질지표를 남기고 정규화된 테이블 카탈로그 생성
2. **Metadata-first 라우팅**: 도메인/기간/엔티티로 후보를 선필터링 후 벡터 검색
3. **Lazy Engine + Cache**: 선택된 테이블만 엔진 생성
4. **Top-k + 재합성**: 후보 테이블 여러 개를 조회 후 최종 합성
5. **관측성(Observability)**: 라우팅 경로, 지연, 실패율, 비용 로그 축적


## 실무 전략: 전처리 -> 테이블 적재 -> 쿼리 라우팅

### 1) 전처리/적재 케이스 분류
- **Case A. Native Table PDF/HTML/CSV**: 구조 파싱 후 컬럼 표준화
- **Case B. Scan/OCR 문서**: OCR -> 레이아웃 복원 -> 신뢰도(score) 저장
- **Case C. 반정형 문서(보고서/공지)**: 섹션/키밸류 추출 후 사실 테이블화
- **Case D. 이벤트 로그/트랜잭션**: 시계열 파티션, 중복제거, 스키마 버저닝

### 2) 테이블 모델링 원칙
- `raw -> curated -> serving` 3계층으로 분리
- serving 계층은 **질의 단위**로 설계 (예: `billionaire_profile`, `billionaire_yearly_rank`)
- 모든 테이블에 최소 메타데이터 부착: 
  - `table_id, domain, entities, time_grain, updated_at, quality_score, pii_level`

### 3) 라우팅 전략
1. 쿼리에서 의도/엔티티/기간 추출
2. 카탈로그 메타데이터로 1차 후보 필터
3. 후보만 대상으로 proxy 벡터 검색(top-k)
4. 선택 테이블 실행 후 답변 합성
5. 라우팅 로그 저장(후속 튜닝 데이터)

> 핵심은 "문서를 바로 검색"이 아니라, **정규화된 데이터 자산(테이블+카탈로그)을 먼저 만든 뒤 라우팅**하는 것이다.


In [ ]:
from __future__ import annotations

import re
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import pandas as pd
from llama_index.core import Settings, VectorStoreIndex
from llama_index.core.schema import IndexNode

try:
    from llama_index.experimental.query_engine import PandasQueryEngine
except Exception as e:
    raise ImportError(
        "PandasQueryEngine import failed. Install llama-index-experimental or run earlier setup cells."
    ) from e


# -----------------------------
# 1) Catalog build utilities
# -----------------------------
def _infer_domain_from_columns(columns: List[str]) -> str:
    text = " \".join([str(c).lower() for c in columns])
    rules = {
        "finance": ["revenue", "profit", "asset", "market", "net", "stock", "worth"],
        "people": ["name", "age", "rank", "person", "ceo", "founder"],
        "geo": ["country", "city", "region", "state", "nation"],
        "time": ["date", "year", "month", "quarter"],
    }
    for domain, kws in rules.items():
        if any(k in text for k in kws):
            return domain
    return "other"


def _case_type(df: pd.DataFrame) -> str:
    cols = [str(c).lower() for c in df.columns]
    n_rows, n_cols = df.shape

    if n_rows == 0 or n_cols == 0:
        return "empty_or_invalid"
    if any("date" in c or "year" in c for c in cols):
        return "time_series_table"
    if any("name" in c or "rank" in c for c in cols):
        return "entity_profile_table"
    if n_cols > 20:
        return "wide_analytical_table"
    return "generic_structured_table"


def build_table_catalog(table_dfs: List[pd.DataFrame], source_id: str = "pdf_wiki_billionaires") -> pd.DataFrame:
    rows = []
    now = pd.Timestamp.utcnow()

    for i, df in enumerate(table_dfs):
        safe_df = df.copy()
        safe_df.columns = [str(c).strip() for c in safe_df.columns]
        cols = list(safe_df.columns)

        domain = _infer_domain_from_columns(cols)
        case_type = _case_type(safe_df)
        null_ratio = float(safe_df.isna().mean().mean()) if len(safe_df) else 1.0
        quality_score = max(0.0, min(1.0, 1.0 - null_ratio))

        table_id = f"tbl_{source_id}_{i:04d}"
        profile_text = (
            f"table_id={table_id}; source={source_id}; domain={domain}; case_type={case_type}; "
            f"rows={len(safe_df)}; cols={len(cols)}; columns={', '.join(cols[:25])}"
        )

        rows.append(
            {
                "table_id": table_id,
                "source_id": source_id,
                "table_name": f"{source_id}_table_{i}",
                "domain": domain,
                "case_type": case_type,
                "row_count": int(len(safe_df)),
                "col_count": int(len(cols)),
                "columns": cols,
                "quality_score": quality_score,
                "updated_at": now,
                "profile_text": profile_text,
            }
        )

    return pd.DataFrame(rows)


# ----------------------------------------
# 2) Production-like Router (Practical)
# ----------------------------------------
@dataclass
class RoutingResult:
    answer: str
    candidate_table_ids: List[str]
    selected_table_ids: List[str]
    partial_answers: List[Tuple[str, str]]
    latency_sec: float


class EnterpriseTableRouter:
    """
    메타데이터 기반 1차 필터 + proxy 벡터 라우팅 + lazy 엔진 생성 + 답변 합성
    """

    def __init__(self, table_dfs: List[pd.DataFrame], catalog_df: pd.DataFrame):
        self.table_dfs = table_dfs
        self.catalog = catalog_df.copy()

        # table_id -> df index 연결
        self.table_id_to_df_idx = {
            row.table_id: int(i) for i, row in self.catalog.reset_index(drop=True).iterrows()
        }

        # proxy index
        proxy_nodes = [
            IndexNode(text=row.profile_text, index_id=row.table_id)
            for _, row in self.catalog.iterrows()
        ]
        self.proxy_index = VectorStoreIndex(proxy_nodes)

        # lazy engine cache
        self._engine_cache: Dict[str, PandasQueryEngine] = {}

    def _query_features(self, query: str) -> Dict[str, str]:
        q = query.lower()
        domain = "other"
        if re.search(r"revenue|profit|asset|market|worth|부자|자산", q):
            domain = "finance"
        elif re.search(r"name|age|rank|who|사람|누구|순위", q):
            domain = "people"
        elif re.search(r"country|city|region|국가|도시", q):
            domain = "geo"
        return {"domain": domain}

    def _metadata_filter(self, query: str, min_quality: float = 0.4) -> List[str]:
        feat = self._query_features(query)
        cand = self.catalog[self.catalog["quality_score"] >= min_quality]

        if feat["domain"] != "other":
            narrowed = cand[cand["domain"] == feat["domain"]]
            if len(narrowed) > 0:
                cand = narrowed

        return cand["table_id"].tolist()

    def _route_topk(self, query: str, candidate_ids: List[str], top_k: int = 3) -> List[str]:
        retriever = self.proxy_index.as_retriever(similarity_top_k=max(top_k * 4, top_k))
        hits = retriever.retrieve(query)

        ranked = []
        allow = set(candidate_ids)
        for h in hits:
            node = getattr(h, "node", h)
            tid = getattr(node, "index_id", None)
            if tid and tid in allow and tid not in ranked:
                ranked.append(tid)
            if len(ranked) >= top_k:
                break

        # fallback
        if not ranked:
            ranked = candidate_ids[:top_k]

        return ranked

    def _get_engine(self, table_id: str) -> PandasQueryEngine:
        if table_id not in self._engine_cache:
            df_idx = self.table_id_to_df_idx[table_id]
            self._engine_cache[table_id] = PandasQueryEngine(
                df=self.table_dfs[df_idx],
                llm=Settings.llm,
                verbose=False,
            )
        return self._engine_cache[table_id]

    def query(self, query: str, top_k: int = 3, debug: bool = True) -> RoutingResult:
        t0 = time.time()

        candidate_ids = self._metadata_filter(query)
        selected_ids = self._route_topk(query, candidate_ids, top_k=top_k)

        partial = []
        for tid in selected_ids:
            try:
                ans = self._get_engine(tid).query(query)
                partial.append((tid, str(ans)))
            except Exception as e:
                partial.append((tid, f"[ERROR] {e}"))

        context = "\n\n".join([f"[table_id={tid}]\n{text}" for tid, text in partial])
        prompt = (
            "You are a data synthesis assistant.\n"
            "Create one final answer from partial table answers.\n"
            "If evidence conflicts, explicitly mention conflicts and confidence.\n\n"
            f"User Query: {query}\n\n"
            f"Partial Answers:\n{context}"
        )

        final = Settings.llm.complete(prompt)
        final_text = getattr(final, "text", str(final))
        latency = time.time() - t0

        if debug:
            print("candidate_table_ids:", candidate_ids[:10], f"... total={len(candidate_ids)}")
            print("selected_table_ids:", selected_ids)
            print("lazy_cache_size:", len(self._engine_cache))
            print("latency_sec:", round(latency, 3))

        return RoutingResult(
            answer=final_text,
            candidate_table_ids=candidate_ids,
            selected_table_ids=selected_ids,
            partial_answers=partial,
            latency_sec=latency,
        )


# ----------------------------------------
# 3) Run with existing notebook variables
# ----------------------------------------
if "table_dfs" not in globals():
    raise ValueError("table_dfs not found. Run table extraction cells first.")

catalog_df = build_table_catalog(table_dfs=table_dfs, source_id="billionaire_pdf")
display(catalog_df[["table_id", "domain", "case_type", "row_count", "col_count", "quality_score"]].head())

enterprise_router = EnterpriseTableRouter(table_dfs=table_dfs, catalog_df=catalog_df)

sample_query = "가장 부자인 사람은 누구이고 어느 국가 출신이야?"
result = enterprise_router.query(sample_query, top_k=3, debug=True)
print(result.answer)
